In [138]:
from __future__ import annotations

In [139]:
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [140]:
import numpy as np
from typing import Any, Optional

from minitorch.tensor.tensor import Tensor
from minitorch.attention.attention import MultiHeadAttention
from minitorch.tokenization.tokenizer import CharTokenizer, BPETokenizer
from minitorch.dataloaders.dataloader import DataLoader
from minitorch.embendding.embed import EmbeddingLayer, Embedding, PositionalEncoding
from minitorch.losses.losses import MSE, SoftMaxCrossEntropy, BCEWithLogits, log_softmax
from minitorch.nn.layers import Linear, Module, LayerNormalization, Sequential, Residual, Dropout
from minitorch.activations.activations import Softmax, GELU, ReLU
from minitorch.optimizers.optim import AdamW
from minitorch.transformer.transformer import GPT

In [141]:
text_path = "C:\\Users\\User\\Downloads\\BIASHARA_Cleaned.txt"

with open(text_path, 'r', encoding='utf-8') as f:
    text = f.read()

In [142]:
#* split into training and val sets
n = 0.95
text_len = len(text)

chars = sorted(set(text))

context_length, batch_size, embed_dim = 8,8,32
#* tokenize the text
tokenizer = CharTokenizer() #* character level tokenization
tokenizer.build_vocab([text])
vocab_size = tokenizer.vocab_size

In [143]:
data = Tensor(tokenizer.encode(text))
train_data = data[:int(n*data.shape[0])]
val_data = data[int(n*data.shape[0]):]
train_loader = DataLoader(train_data, context_length)
val_loader = DataLoader(val_data, context_length)


In [144]:
def evaluate_loss(
    train_data: list[Tensor],
    val_data: list[Tensor],
    model: GPT,
    eval_iters: int) -> dict[str, Any]:
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = np.zeros(shape=(eval_iters))
        for i in range(eval_iters):
            if split == 'train':
                xs,ys = train_data
                _, loss = model.forward(xs,ys)
                losses[i] = loss.data
            else:
                xs,ys = val_data
                _, loss = model.forward(xs,ys)
                losses[i] = loss.data
        out[split] = losses.mean()
    model.train()
    return out

In [149]:

###############################################################
#* define the hyperparameters
context_length, batch_size, embed_dim = 8,32,64
config = {
    'learning_rate': 1e-4,
    'embed_dim': embed_dim,
    'vocab_size': vocab_size,
    'max_seq_length': 1024,
    'n_heads': 8,
    'n_layers': 6,
    'dim': embed_dim,
    'dropout': 0.1,
    'max_iters': 10000,
    'eval_iters': 1000,
    
}

#* ###########################################################
gpt = GPT(config)                                 #* model
params = gpt.parameters()                         #* parameters
optimizer = AdamW(params, lr=config['learning_rate'], eps=1e-9, betas=(0.9,0.98))                #* optimizer

#* ###########################################################
#* training loop
print("Epoch            | Training Loss         | Validation Loss | ")
print("-" * 60)

gpt.train()
for i in range(config['max_iters']):
    optimizer.zero_grad()
    
    #* get fresh batches each iteration
    train_xs, train_ys = train_loader.get_batch(batch_size)
    val_xs, val_ys = val_loader.get_batch(batch_size)
    
    #* zero grad all the parameters and ran evaluation
    if (i) % config['eval_iters'] == 0:
        losses = evaluate_loss([train_xs, train_ys],[val_xs, val_ys] , gpt, config['eval_iters'])
        print(f'{i+1}           |          {losses["train"]:4f}         |           {losses["val"]:4f}')
        
    #* train the model(forward pass ->> backward pass)
    _, loss = gpt.forward(train_xs, train_ys)
    loss.backward()

    # * update the model parameters
    optimizer.step()

Epoch            | Training Loss         | Validation Loss | 
------------------------------------------------------------
1           |          4.042758         |           4.135287
1001           |          3.399707         |           3.398162
2001           |          3.409364         |           3.409466
3001           |          3.438868         |           3.439833
4001           |          3.508321         |           3.510127
5001           |          3.450865         |           3.454063
6001           |          3.246089         |           3.246601
7001           |          3.029108         |           3.076783
8001           |          2.966599         |           2.950008
9001           |          2.924748         |           2.872696


In [150]:
loss

Tensor(data=2.7712502479553223,shape=(), requires_grad= True)

In [148]:
total_params = 0.0

for param in gpt.parameters():
    if isinstance(param, list):
        for p in param:
            if isinstance(p, list):
                for sub_p in p:
                    total_params += np.prod(sub_p.shape)
            else:
                total_params += np.prod(p.shape)
    else:
        total_params += np.prod(param.shape)
        
total_params = total_params / 1e6  # Convert to millions
print(f"Total parameters: {total_params:.2f} M")

Total parameters: 0.36 M


In [147]:
params

[Tensor(data=[[ 0.01448496 -0.04178081  0.10352349 ... -0.06729821 -0.23088254
   -0.22007376]
  [ 0.13015726  0.02365449 -0.09379592 ...  0.06608129 -0.06182367
   -0.18716624]
  [-0.2455183   0.14750801  0.15825734 ... -0.0125033  -0.05755021
   -0.06939271]
  ...
  [ 0.03319653 -0.24372546 -0.25395995 ...  0.04217694  0.12501302
   -0.12334314]
  [-0.06436054 -0.2333152   0.04176784 ...  0.02749629 -0.23092796
   -0.19439317]
  [ 0.13422973 -0.2152398   0.02128409 ... -0.06598151 -0.17652765
   -0.19426711]],shape=(30, 64), requires_grad= True),
 Tensor(data=[[ 0.09295467  0.08195178 -0.15258025 ... -0.1487204  -0.17415172
    0.00326412]
  [-0.00023837  0.10799229  0.10666424 ...  0.1348364  -0.04302713
    0.07230842]
  [ 0.03581326 -0.12742323 -0.15970288 ...  0.09356271 -0.16497684
    0.05583916]
  ...
  [-0.16924778  0.12793563 -0.11615182 ... -0.09136105  0.02624929
   -0.09215644]
  [-0.07692713 -0.16439342  0.08127498 ... -0.09867188 -0.02980255
   -0.13743411]
  [ 0.010090